# 6/7 - H. Clips
Clip planner (best event per metric / subtype / team / zone x channel cell), annotated video + animated top-down renderer, stoppage check sheets.

*Original sections: sections 19c, 19d, 19e.*  Barca (team 0) vs Atletico (team 1), first half.

**Run the notebooks in order** (each one reads what the earlier ones saved in `new_cache/pressure_analysis/state/`):

| # | notebook | what it covers |
|---|---|---|
| 1 | 01_pressure_signal.ipynb | B. The pressure signal |
| 2 | 02_where_zone_channel_line.ipynb | C. WHERE - zone, channel, defensive line, shape |
| 3 | 03_timing_and_possession.ipynb | D. WHEN - reaction speed and possession-level pressure |
| 4 | 04_passes_outcomes_triggers.ipynb | E/F. EFFECT and TRIGGERS |
| 5 | 05_summary_and_registry.ipynb | G/H. Summary + event registry |
| 6 | **06_clips.ipynb** | H. Clips |
| 7 | 07_exports.ipynb | I. Exports |

Shared setup (paths, tables, data contract, team-label orientation, helpers) lives in `pressure_common.py` - keep it in the same folder as the notebooks.

In [ ]:
from pressure_common import *

In [ ]:
# handed over by the earlier notebooks:
#   05_summary_and_registry: EVENTS
#   04_passes_outcomes_triggers: PRESS_ONSET_WINDOW_FRAMES
#   01_pressure_signal: lane_status, off_ball_pressure, stoppages
load_state(globals(), [
    'EVENTS',
    'PRESS_ONSET_WINDOW_FRAMES',
    'lane_status',
    'off_ball_pressure',
    'stoppages',
])

## 19c. Clip planner - pick the best clip for every metric / subtype / team (and every zone x channel cell)
Selection is automatic (no manual labeling): inside each group the highest-`score` event wins, subject to a minimum gap between clips of the same team so the set is varied. Look at the printed plan first - set `RENDER_FAMILIES` to render only some families, `N_PER_GROUP` for more clips per group, or `MANUAL_UIDS` to force specific events from `EVENTS`.

The groups are what answers the structure questions directly: **zone x channel x team** (where it presses), **target** (on whom), **defensive line** (high / deep / step-up), **counter-press** speed, **pass outcomes under pressure**, **triggers**, **shape**, **tackles / interceptions**, **possessions**.


In [ ]:
# ===== 19c. Clip planner =====
N_PER_GROUP = 1
MIN_SEP_SEC = 8.0
CLIP_BUDGET = 48
FAMILY_CAP = {"pressure_event": 10}          # max clips per family (default below)
DEFAULT_FAMILY_CAP = 4
RENDER_FAMILIES = None                        # e.g. ["pressure_event", "defensive_line"]; None = all
MANUAL_UIDS = []                              # force specific EVENTS.uid values
MAX_EVENT_CLIP_SEC = 14.0
PAD_BEFORE_SEC, PAD_AFTER_SEC = 1.5, 1.5

GROUPS = [  # (label, family, subtypes or None, group-by columns)
    ("press: zone x channel",     "pressure_event",      ["coordinated_press"],                   ["team", "zone", "channel"]),
    ("press: on whom",            "pressure_event",      None,                                   ["team", "target"]),
    ("press: lone / lane-block",  "pressure_event",      ["lone_chaser", "lane_block_press"],     ["team", "subtype"]),
    ("press won the ball",        "pressure_regain",     None,                                   ["team", "zone"]),
    ("defensive line",            "defensive_line",      None,                                   ["team", "subtype"]),
    ("team shape",                "team_shape",          None,                                   ["team", "subtype"]),
    ("counter-press speed",       "counter_press",       None,                                   ["team", "subtype"]),
    ("passes under pressure",     "pass_under_pressure", None,                                   ["team", "subtype"]),
    ("long ball lost",            "long_ball",           None,                                   ["team", "subtype"]),
    ("tackles / interceptions",   "defensive_action",    None,                                   ["team", "subtype"]),
    ("press triggers",            "press_trigger",       None,                                   ["team", "subtype"]),
    ("possession pressure",       "possession_pressure", None,                                   ["team", "subtype"]),
]

def _overlap(a0, a1, b0, b1):
    return max(0, min(a1, b1) - max(a0, b0)) / max(1, min(a1 - a0, b1 - b0))

def _window(row):
    fs = int(row.start_frame) - int(PAD_BEFORE_SEC * FPS)
    fe = min(int(row.end_frame), int(row.start_frame) + int(MAX_EVENT_CLIP_SEC * FPS)) + int(PAD_AFTER_SEC * FPS)
    if row.family in ("pass_under_pressure", "press_trigger", "defensive_action", "long_ball"):      # point events: centre on the anchor
        fs = int(row.anchor_frame) - int(2.0 * FPS); fe = max(fe, int(row.anchor_frame) + int(3.5 * FPS))
    if row.family == "counter_press":
        fs = int(row.payload["loss_frame"]) - int(1.5 * FPS)
    return max(0, fs), min(int(player_frame_table["frame_idx"].max()), fe)

chosen, per_family = [], {}
def _take(row, label):
    fs, fe = _window(row)
    for c in chosen:
        if c["team"] == row.team and _overlap(fs, fe, c["fs"], c["fe"]) > 0.3:
            return False
        if c["team"] == row.team and abs(fs - c["fs"]) < MIN_SEP_SEC * FPS and c["family"] == row.family and c["subtype"] == row.subtype:
            return False
    chosen.append(dict(uid=row.uid, group=label, family=row.family, subtype=row.subtype, team=int(row.team), zone=row.zone, channel=row.channel,
                       target=row.target, score=row.score, anchor=int(row.anchor_frame), fs=fs, fe=fe, payload=row.payload))
    per_family[row.family] = per_family.get(row.family, 0) + 1
    return True

for uid in MANUAL_UIDS:
    m = EVENTS[EVENTS.uid == uid]
    if len(m): _take(next(m.itertuples()), "manual")

for label, fam, subs, by in GROUPS:
    if RENDER_FAMILIES is not None and fam not in RENDER_FAMILIES:
        continue
    d = EVENTS[EVENTS.family == fam]
    if subs: d = d[d.subtype.isin(subs)]
    d = d.dropna(subset=[c for c in by if c != "team"])
    for _, g in d.groupby(by):
        taken = 0
        for row in g.sort_values("score", ascending=False).itertuples():
            if len(chosen) >= CLIP_BUDGET or per_family.get(fam, 0) >= FAMILY_CAP.get(fam, DEFAULT_FAMILY_CAP):
                break
            if _take(row, label):
                taken += 1
            if taken >= N_PER_GROUP:
                break

CLIP_PLAN = pd.DataFrame(chosen)
if len(CLIP_PLAN):
    CLIP_PLAN["team_name"] = CLIP_PLAN["team"].map(TEAM_NAMES)
    CLIP_PLAN["clock"] = CLIP_PLAN["anchor"].map(clock_str)
    CLIP_PLAN["dur_sec"] = (CLIP_PLAN["fe"] - CLIP_PLAN["fs"]) / FPS
    CLIP_PLAN["clip_id"] = [f"{i:02d}_{r.team_name}_{r.family}_{r.subtype}_{r.clock.replace(':', 'm')}s" for i, r in enumerate(CLIP_PLAN.itertuples())]
    print(f"{len(CLIP_PLAN)} clips planned, ~{CLIP_PLAN.dur_sec.sum() / 60:.1f} min of video\n")
    print(CLIP_PLAN[["clip_id", "group", "zone", "channel", "target", "score", "dur_sec"]].round(1).to_string(index=False))
else:
    print("no clips planned - check the event inventory above")


## 19d. Render the clips (annotated video + animated top-down view)
Every clip is shown from the **focus team's** point of view (the team pressing / defending). Left: `annotated_teams.mp4` with the metric layer on top. Right: an animated top-down panel in the focus team's frame (it always attacks to the right), with the zone / channel grid, the defensive line, the opponent block and the same event marks.

**What is drawn (video and panel)**
- orange rings + lines = **pressers** (exactly the `is_pressuring` players from section 1), small yellow triangle above the head = **ball carrier**
- carrier -> teammate lines = **passing lanes** from section 2: red = tight-blocked, amber dashed = contested, green = open
- cyan polyline = the focus team's **defensive line** (last-3), with its height in metres; thin on pressing clips, thick on defensive-line / shape clips
- 3 x 3 **zone x channel grid** on the panel with the ball's cell highlighted; zone / channel / target in the HUD
- event marks: pass arrow + outcome (passes under pressure, long balls), cyan ring + trigger name (press triggers), burst + label (tackle / interception), LOSS -> FIRST PRESS -> REGAINED banners (counter-press), hull + depth / width (team shape)
- bottom of the panel: **pressers over time** with the event window shaded and a moving cursor

Everything is in `STYLE` (colours, thickness) so the look can be changed in one place; `save_style` / `load_style` keep it as json. Rendering is incremental: existing mp4s are skipped unless `OVERWRITE = True`.


In [ ]:
# ===== 19d. Clip renderer =====
import cv2, shutil, subprocess, time, json
from scipy.spatial import ConvexHull
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.patches import Rectangle, Circle

# ---------------- settings ----------------
RENDER_MODE = "both"        # "both" | "overlay" | "tactical"
OUT_STRIDE = 2              # write every 2nd frame (12.5 fps) - 2x faster; 1 = full rate
OVERWRITE = False
TRACK_W, TRACK_H = 1920, 1080      # pixel space of x1,y1,x2,y2,foot_px_*
VIDEO_FRAME_OFFSET = 0             # video frame = frame_idx + offset
DRAW_NUMBERS = False               # annotated_teams.mp4 is probably already labelled
OUT_H, VIDEO_W, PANEL_W = 720, 1280, 640

STYLE = {
    "bgr": dict(presser=(0, 150, 255), carrier=(0, 230, 255), lane_tight=(60, 60, 235), lane_contested=(0, 200, 255), lane_open=(80, 220, 80),
                line=(255, 210, 0), event=(255, 60, 200), white=(255, 255, 255)),
    "panel": dict(focus="#ff9f43", other="#4da3ff", line="#00d2ff", ring="#ffe600"),
}
def save_style(path): Path(path).write_text(json.dumps(STYLE, indent=2))
def load_style(path):
    d = json.loads(Path(path).read_text())
    for k, v in d.items(): STYLE[k].update({a: tuple(b) if isinstance(b, list) else b for a, b in v.items()})
BGR = STYLE["bgr"]

# ---------------- fast slices of the notebook's own tables ----------------
_pcols = [c for c in ["frame_idx", "track_id", "role", "team", "pitch_x", "pitch_y", "x1", "y1", "x2", "y2", "foot_px_x", "foot_px_y"] if c in player_frame_table.columns]
_PFT = player_frame_table.loc[player_frame_table["team"].notna(), _pcols].sort_values("frame_idx").reset_index(drop=True)
_OBP = off_ball_pressure[["frame_idx", "track_id", "team", "is_pressuring"]].sort_values("frame_idx").reset_index(drop=True)
_LANE = lane_status.sort_values("frame_idx").reset_index(drop=True)
_BALL = ball_frame_table.sort_values("frame_idx").reset_index(drop=True)
_PFT_F, _OBP_F, _LANE_F, _BALL_F = (d["frame_idx"].values for d in (_PFT, _OBP, _LANE, _BALL))
def _slice(df, F, f0, f1):
    return df.iloc[np.searchsorted(F, f0, "left"): np.searchsorted(F, f1, "right")]
def to_view(x, y, F):
    # absolute pitch -> frame in which team F attacks to the right (own goal at x = 0)
    return (x, y) if ATTACK_SIGN[int(F)] == 1.0 else (PITCH_LENGTH - x, PITCH_WIDTH - y)
_clock_f = clock_str
# ---------------- per-frame state for one clip ----------------
def make_states(spec):
    F, fs, fe = int(spec["team"]), int(spec["fs"]), int(spec["fe"])
    pf, ob, ln = _slice(_PFT, _PFT_F, fs, fe), _slice(_OBP, _OBP_F, fs, fe), _slice(_LANE, _LANE_F, fs, fe)
    bl = _slice(_BALL, _BALL_F, fs, fe).set_index("frame_idx")
    car, cteam = bl["carrier_track_id"].ffill(limit=12), bl["carrier_team"].ffill(limit=12)      # display only: bridge carrier flicker
    press = ob[ob["is_pressuring"].astype(bool) & (ob["team"] == F)].groupby("frame_idx")["track_id"].apply(lambda s: s.astype(int).values).to_dict()
    lanes = {int(f): dict(zip(g["teammate_id"].astype(int), np.where(g["lane_tight_block"], "tight", np.where(g["lane_contested"], "contested", "open"))))
             for f, g in ln.groupby("frame_idx")}
    states = {}
    for f, g in pf.groupby("frame_idx"):
        f = int(f); g = g.drop_duplicates("track_id"); gi = g.set_index("track_id", drop=False)
        vx, vy = to_view(g["pitch_x"].values, g["pitch_y"].values, F)
        st = dict(f=f, rows=gi, ids=g["track_id"].values.astype(int), team=g["team"].values.astype(int), role=g["role"].values, vxy=np.c_[vx, vy])
        c = car.get(f, np.nan); ct = cteam.get(f, np.nan)
        st["car"] = None if pd.isna(c) else int(c); st["car_team"] = None if pd.isna(ct) else int(ct)
        bx, by = (bl.at[f, "ball_pitch_x"], bl.at[f, "ball_pitch_y"]) if f in bl.index else (np.nan, np.nan)
        st["ball_abs"] = None if pd.isna(bx) else (float(bx), float(by)); st["ball"] = None if pd.isna(bx) else np.array(to_view(float(bx), float(by), F))
        st["zone"] = zone_for_team(bx, F) if pd.notna(bx) else None; st["channel"] = channel_for_team(by, F) if pd.notna(by) else None
        mF = (st["team"] == F) & (st["role"] == "player"); fx, fid = st["vxy"][mF], st["ids"][mF]
        st.update(line_ids=np.array([], int), front_ids=np.array([], int), line_h=np.nan, depth=np.nan, width=np.nan, area=np.nan, hull=None)
        if len(fx) >= 6:
            o = np.argsort(fx[:, 0]); st["line_ids"], st["front_ids"] = fid[o[:3]], fid[o[-3:]]
            st["line_h"] = float(fx[o[:3], 0].mean()); st["depth"] = float(fx[o[-3:], 0].mean() - fx[o[:3], 0].mean())
            st["width"] = team_frame_width(fx[:, 1]); st["area"] = st["depth"] * st["width"]
            if len(fx) >= 3: st["hull"] = fx[ConvexHull(fx).vertices]
        st["pressers"] = press.get(f, np.array([], int)); st["lanes"] = lanes.get(f, {})
        st["n_press"] = len(st["pressers"]); st["n_lane"] = len(st["lanes"]); st["n_tight"] = sum(v == "tight" for v in st["lanes"].values())
        states[f] = st
    # event helpers computed once per clip
    pay = spec["payload"]
    spec["_regain_frame"] = None
    if spec["family"] == "counter_press":
        raw = bl[(bl.index > pay["gain_frame"]) & (bl["carrier_team"] == F)]
        spec["_regain_frame"] = int(raw.index[0]) if len(raw) else None
        tt = pay.get("time_to_press_sec"); spec["_first_press_frame"] = None if tt is None else int(pay["loss_frame"] + tt * FPS)
    spec["_pass_line"] = None
    if "release_frame" in pay and "reception_frame" in pay and "passer_track_id" in pay:
        s0, s1 = states.get(int(pay["release_frame"])), states.get(int(pay["reception_frame"]))
        def _pos(s, t):
            return None if s is None or t not in list(s["ids"]) else s["vxy"][list(s["ids"]).index(t)]
        a, b = (_pos(s0, pay["passer_track_id"]), _pos(s1, pay["receiver_track_id"]))
        spec["_pass_line"] = None if a is None or b is None else (a, b)
    return states

# ---------------- text shown on both views ----------------
def hud_lines(spec, st, f):
    fam, sub, F, pay = spec["family"], spec["subtype"], int(spec["team"]), spec["payload"]
    z = (st or {}).get("zone") or spec.get("zone") or "?"; ch = (st or {}).get("channel") or spec.get("channel") or "?"
    l1 = f"{TEAM_NAMES[F]} | {fam.replace('_', ' ')}: {sub.replace('_', ' ')}"
    l2 = f"{_clock_f(f)}  t{(f - spec['anchor']) / FPS:+.1f}s   zone {z.replace('_', ' ')} | channel {ch}" + (f" | on {spec['target']}" if spec.get("target") and fam in ("pressure_event", "pressure_regain") else "")
    if st is None:
        return [l1, l2, "", ""]
    if fam == "team_shape":
        l3 = f"depth {st['depth']:.0f} m  width {st['width']:.0f} m  area {st['area']:.0f} m2"
    elif fam == "defensive_line":
        l3 = f"line {st['line_h']:.0f} m from own goal (last-3)   pressers {st['n_press']}"
    else:
        l3 = f"pressers {st['n_press']}   lanes blocked {st['n_tight']}/{st['n_lane']}   line {st['line_h']:.0f} m"
    l4 = ""
    if fam in ("pass_under_pressure",):
        pm = pay.get("progression_m"); l4 = f"pass {pay['pass_distance_m']:.0f} m -> {pay['outcome'].replace('_', ' ')}" + ("" if pm is None else f" ({pm:+.0f} m)")
    elif fam == "press_trigger":
        l4 = f"trigger: {pay['trigger'].replace('_', ' ')} | press followed: {'YES' if pay['press_followed'] else 'no'}"
    elif fam == "defensive_action":
        l4 = pay["action_type"].upper()
    elif fam == "long_ball":
        l4 = "long ball lost under pressure"
    elif fam == "counter_press":
        if f < pay["gain_frame"]: l4 = "BALL LOST"
        else:
            l4 = f"t+{(f - pay['gain_frame']) / FPS:.1f}s since takeover | response: {pay['response']}"
            if spec.get("_first_press_frame") and f >= spec["_first_press_frame"]: l4 += " | FIRST PRESS"
            if spec.get("_regain_frame") and f >= spec["_regain_frame"]: l4 = "REGAINED THE BALL"
    elif fam == "possession_pressure":
        l4 = f"{pay['pct_pressed']:.0%} of this possession pressed"
    elif fam in ("pressure_event", "pressure_regain"):
        l4 = f"max {pay['max_n']:.0f} pressers | " + ("won the ball back" if pay["regained"] else "no regain")
    return [l1, l2, l3, l4]

def _event_active(spec, f):
    # which event mark (if any) is live at frame f
    fam, pay = spec["family"], spec["payload"]
    if fam in ("pass_under_pressure", "long_ball"): return pay["release_frame"] <= f <= pay["reception_frame"] + 20
    if fam == "press_trigger": return pay["reception_frame"] - 5 <= f <= pay["reception_frame"] + PRESS_ONSET_WINDOW_FRAMES
    if fam == "defensive_action": return pay["action_frame"] - 12 <= f <= pay["action_frame"] + 25
    return False

# ---------------- video overlay ----------------
def _pix(st, tid):
    if tid is None or tid not in st["rows"].index or "foot_px_x" not in st["rows"].columns: return None
    r = st["rows"].loc[tid]; v = [r["foot_px_x"], r["foot_px_y"], r["x1"], r["y1"], r["x2"], r["y2"]]
    return None if any(pd.isna(v)) else [float(a) for a in v]

def _dashed(img, p0, p1, color, th, dash=14):
    p0, p1 = np.array(p0, float), np.array(p1, float); n = int(np.linalg.norm(p1 - p0) // dash)
    for i in range(0, n, 2):
        a = p0 + (p1 - p0) * i / n; b = p0 + (p1 - p0) * min(i + 1, n) / n
        cv2.line(img, tuple(a.astype(int)), tuple(b.astype(int)), color, th, cv2.LINE_AA)

def draw_overlay(img, st, f, spec):
    h, w = img.shape[:2]; sx, sy = w / TRACK_W, h / TRACK_H; th = max(1, int(round(2 * w / 1920))); sc = w / 1920
    S = lambda p: (int(p[0] * sx), int(p[1] * sy)); fam, pay = spec["family"], spec["payload"]
    if st is not None:
        car = _pix(st, st["car"]); F = int(spec["team"])
        thick_line = fam in ("defensive_line", "team_shape")
        # defensive line (+ front line for shape clips)
        for ids, lab in [(st["line_ids"], "line"), (st["front_ids"] if fam == "team_shape" else [], "front")]:
            pts = [p for p in (_pix(st, int(i)) for i in ids) if p is not None]
            if len(pts) >= 2:
                pts = sorted(pts, key=lambda p: p[0]); cv2.polylines(img, [np.array([S(p[:2]) for p in pts], np.int32)], False, BGR["line"], th + (3 if thick_line else 0), cv2.LINE_AA)
                if lab == "line" and np.isfinite(st["line_h"]):
                    cv2.putText(img, f"{st['line_h']:.0f} m", (S(pts[0])[0], S(pts[0])[1] + int(26 * sc)), cv2.FONT_HERSHEY_SIMPLEX, max(0.5, 0.8 * sc), BGR["line"], max(1, int(2 * sc)), cv2.LINE_AA)
        if car is not None and st["car_team"] != F:
            for tid, status in st["lanes"].items():
                o = _pix(st, int(tid))
                if o is None: continue
                if status == "tight": cv2.line(img, S(car[:2]), S(o[:2]), BGR["lane_tight"], th + 1, cv2.LINE_AA)
                elif status == "contested": _dashed(img, S(car[:2]), S(o[:2]), BGR["lane_contested"], th)
                else: cv2.line(img, S(car[:2]), S(o[:2]), BGR["lane_open"], max(1, th - 1), cv2.LINE_AA)
            for pid_ in st["pressers"]:
                p = _pix(st, int(pid_))
                if p is None: continue
                cv2.ellipse(img, S(p[:2]), (int(26 * sx), int(9 * sy)), 0, 0, 360, BGR["presser"], th + 1, cv2.LINE_AA)
                cv2.line(img, S(p[:2]), S(car[:2]), BGR["presser"], max(1, th - 1), cv2.LINE_AA)
        if car is not None:
            cx, top = int((car[2] + car[4]) / 2 * sx), int(car[3] * sy)
            tri = np.array([[cx, top - 4 * sy], [cx - 9 * sx, top - 22 * sy], [cx + 9 * sx, top - 22 * sy]], np.int32)
            cv2.fillPoly(img, [tri], BGR["carrier"]); cv2.polylines(img, [tri], True, (0, 0, 0), 1, cv2.LINE_AA)
        if DRAW_NUMBERS:
            for tid in st["rows"].index:
                p = _pix(st, int(tid)); n = TRACK_NUM.get(int(tid))
                if p is not None and n is not None:
                    cv2.putText(img, str(n), (int((p[2] + p[4]) / 2 * sx) - 6, int(p[3] * sy) - 26), cv2.FONT_HERSHEY_SIMPLEX, max(0.35, 0.6 * sc), BGR["white"], 1, cv2.LINE_AA)
        # event marks
        if _event_active(spec, f):
            if fam in ("pass_under_pressure", "long_ball"):
                a, b = _pix(st, pay.get("passer_track_id")), _pix(st, pay.get("receiver_track_id"))
                if a is not None and b is not None:
                    cv2.arrowedLine(img, S(a[:2]), S(b[:2]), BGR["event"], th + 1, cv2.LINE_AA, tipLength=0.06)
            elif fam == "press_trigger":
                r = _pix(st, pay["receiver_track_id"])
                if r is not None:
                    cv2.ellipse(img, S(r[:2]), (int(34 * sx), int(12 * sy)), 0, 0, 360, BGR["event"], th + 2, cv2.LINE_AA)
                    cv2.putText(img, pay["trigger"].replace("_", " "), (S(r[:2])[0] - 40, S(r[:2])[1] + int(34 * sc)), cv2.FONT_HERSHEY_SIMPLEX, max(0.5, 0.8 * sc), BGR["event"], max(1, int(2 * sc)), cv2.LINE_AA)
            elif fam == "defensive_action" and car is not None:
                cv2.circle(img, S(car[:2]), int(30 * sc) + 6, BGR["event"], th + 2, cv2.LINE_AA)
                cv2.putText(img, pay["action_type"].upper(), (S(car[:2])[0] - 50, S(car[:2])[1] + int(55 * sc)), cv2.FONT_HERSHEY_SIMPLEX, max(0.6, 0.9 * sc), BGR["event"], max(1, int(2 * sc)), cv2.LINE_AA)
    lines = [t for t in hud_lines(spec, st, f) if t]
    pad, lh, fs_ = max(6, int(16 * sc)), max(14, int(40 * sc)), max(0.45, 0.9 * sc)
    x0, y0, x1, y1 = pad, pad, pad + int(1250 * sc), pad + lh * len(lines) + pad // 2
    roi = img[y0:y1, x0:x1]; img[y0:y1, x0:x1] = cv2.addWeighted(roi, 0.35, np.zeros_like(roi), 0.65, 0)
    for i, t in enumerate(lines):
        cv2.putText(img, t, (x0 + pad // 2, y0 + lh * (i + 1) - pad // 3), cv2.FONT_HERSHEY_SIMPLEX, fs_, BGR["white"], max(1, int(2 * sc)), cv2.LINE_AA)
    return img

# ---------------- top-down animated panel ----------------
class Panel:
    def __init__(self, states, spec):
        self.S, self.spec = states, spec; P_ = STYLE["panel"]
        self.fig = Figure(figsize=(PANEL_W / 100, OUT_H / 100), dpi=100, facecolor="#111418"); FigureCanvasAgg(self.fig)
        self.ax = self.fig.add_axes([.02, .50, .96, .47]); self.ax.set_facecolor("#2d6a30"); self.ax.set_xlim(-2, PITCH_LENGTH + 2); self.ax.set_ylim(PITCH_WIDTH + 2, -2)
        self.ax.set_aspect("equal"); self.ax.axis("off"); c = "#d8d8d8"; ax = self.ax
        ax.add_patch(Rectangle((-2, -2), PITCH_LENGTH + 4, PITCH_WIDTH + 4, fc="#2d6a30", ec="none", zorder=0))
        ax.add_patch(Rectangle((0, 0), PITCH_LENGTH, PITCH_WIDTH, fill=False, ec=c, lw=1)); ax.plot([PITCH_LENGTH / 2] * 2, [0, PITCH_WIDTH], c=c, lw=1)
        ax.add_patch(Circle((PITCH_LENGTH / 2, PITCH_WIDTH / 2), 9.15, fill=False, ec=c, lw=1))
        for x0, w_ in [(0, 16.5), (PITCH_LENGTH - 16.5, 16.5)]: ax.add_patch(Rectangle((x0, PITCH_WIDTH / 2 - 20.16), w_, 40.32, fill=False, ec=c, lw=1))
        for x in (35, 70): ax.plot([x, x], [0, PITCH_WIDTH], c="#ffffff", lw=.8, ls=":", alpha=.6)                  # zone grid
        for y in (PITCH_WIDTH / 3, 2 * PITCH_WIDTH / 3): ax.plot([0, PITCH_LENGTH], [y, y], c="#ffffff", lw=.8, ls=":", alpha=.6)   # channel grid
        for x, lab in [(17.5, "own third"), (52.5, "middle"), (87.5, "attacking third")]: ax.text(x, -0.8, lab, color=c, fontsize=6.5, ha="center", va="bottom")
        for y, lab in [(PITCH_WIDTH / 6, "L"), (PITCH_WIDTH / 2, "C"), (5 * PITCH_WIDTH / 6, "R")]: ax.text(-1.6, y, lab, color=c, fontsize=7, ha="right", va="center")
        self.at = self.fig.add_axes([.04, .17, .92, .30]); self.at.axis("off"); self.at.set_xlim(0, 1); self.at.set_ylim(0, 1)
        self.al = self.fig.add_axes([.09, .04, .86, .10], facecolor="#1b1f26")
        fl = np.array(sorted(states)); npz = np.array([states[f]["n_press"] for f in fl])
        self.al.fill_between(fl, npz, step="post", color=P_["focus"], alpha=.85); self.al.set_ylim(0, max(3, npz.max() + 1)); self.al.set_xlim(fl.min(), fl.max())
        self.al.axvspan(spec["anchor"] - 1, spec["anchor"] + 1, color="white", alpha=.15)
        self.al.set_xticks([]); self.al.tick_params(colors="#aaa", labelsize=7); self.al.set_ylabel("pressers", color="#aaa", fontsize=7)
        self.cur = self.al.axvline(fl.min(), c="#ff4d4d", lw=1.5); self.dyn = []
    def _add(self, a): self.dyn.extend(a if isinstance(a, (list, tuple)) else [a])
    def render(self, f):
        for a in self.dyn: a.remove()
        self.dyn = []; ax = self.ax; st = self.S.get(f); spec = self.spec; P_ = STYLE["panel"]; F = int(spec["team"]); fam, pay = spec["family"], spec["payload"]
        self.cur.set_xdata([f, f])
        if st is not None:
            if st["ball"] is not None and st["zone"]:                                    # highlight the ball's zone x channel cell
                zi = ["own_third", "middle_third", "attacking_third"].index(st["zone"]); ci = ["left", "centre", "right"].index(st["channel"])
                self._add(ax.add_patch(Rectangle((35 * zi if zi else 0, PITCH_WIDTH / 3 * ci), 35, PITCH_WIDTH / 3, fc="white", alpha=.10, ec="none", zorder=1)))
            if st["hull"] is not None and len(st["hull"]) >= 3: self._add(ax.fill(st["hull"][:, 0], st["hull"][:, 1], color=P_["focus"], alpha=.12, zorder=1))
            lp = [(st["vxy"][list(st["ids"]).index(i)]) for i in st["line_ids"] if i in list(st["ids"])]
            if len(lp) >= 2:
                lp = np.array(sorted(lp, key=lambda p: p[1])); thick = fam in ("defensive_line", "team_shape")
                self._add(ax.plot(lp[:, 0], lp[:, 1], c=P_["line"], lw=3.2 if thick else 1.6, zorder=3, alpha=.95))
                if np.isfinite(st["line_h"]): self._add(ax.text(lp[:, 0].mean(), lp[:, 1].min() - 2.2, f"{st['line_h']:.0f} m", color=P_["line"], fontsize=8, ha="center", weight="bold", zorder=6))
            if fam == "team_shape":
                fp = [(st["vxy"][list(st["ids"]).index(i)]) for i in st["front_ids"] if i in list(st["ids"])]
                if len(fp) >= 2: fp = np.array(sorted(fp, key=lambda p: p[1])); self._add(ax.plot(fp[:, 0], fp[:, 1], c=P_["line"], lw=3.2, ls="--", zorder=3))
            if st["ball"] is not None and st["car_team"] != F:
                for tid, status in st["lanes"].items():
                    if tid in list(st["ids"]):
                        o = st["vxy"][list(st["ids"]).index(tid)]; col = {"tight": "#ff4d4d", "contested": "#ffc800", "open": "#7cfc00"}[status]
                        self._add(ax.plot([st["ball"][0], o[0]], [st["ball"][1], o[1]], c=col, lw=1.5 if status != "open" else 1.0, ls="-" if status == "tight" else (":" if status == "contested" else "-"), alpha=.9, zorder=3))
            isF = st["team"] == F; gk = np.array([r == "goalkeeper" for r in st["role"]])
            for m, colr in [(isF, P_["focus"]), (~isF, P_["other"])]:
                for sel, mk in [(m & ~gk, "o"), (m & gk, "s")]:
                    if sel.any(): self._add(ax.scatter(st["vxy"][sel, 0], st["vxy"][sel, 1], s=165, c=colr, ec="white", lw=1, marker=mk, zorder=4))
            for i, xy in zip(st["ids"], st["vxy"]):
                n = TRACK_NUM.get(int(i))
                if n is not None: self._add(ax.text(xy[0], xy[1], str(n), ha="center", va="center", fontsize=6.5, color="white", weight="bold", zorder=5))
            for pid_ in st["pressers"]:
                if pid_ in list(st["ids"]): self._add(ax.scatter(*st["vxy"][list(st["ids"]).index(pid_)], s=420, facecolors="none", edgecolors=P_["ring"], lw=1.8, zorder=5))
            if st["car"] is not None and st["car"] in list(st["ids"]): self._add(ax.scatter(*st["vxy"][list(st["ids"]).index(st["car"])], s=380, facecolors="none", edgecolors="white", lw=2.2, zorder=5))
            if st["ball"] is not None: self._add(ax.scatter(*st["ball"], s=60, c="#ffe600", ec="black", zorder=6))
            if _event_active(spec, f):
                if spec.get("_pass_line") is not None:
                    a, b = spec["_pass_line"]; self._add(ax.annotate("", xy=tuple(b), xytext=tuple(a), arrowprops=dict(arrowstyle="-|>", color="#ff3cc8", lw=2.6), zorder=7))
                if fam == "press_trigger" and pay["receiver_track_id"] in list(st["ids"]):
                    self._add(ax.scatter(*st["vxy"][list(st["ids"]).index(pay["receiver_track_id"])], s=560, facecolors="none", edgecolors="#ff3cc8", lw=2.6, zorder=7))
                if fam == "defensive_action" and st["ball"] is not None:
                    self._add(ax.scatter(*st["ball"], s=700, facecolors="none", edgecolors="#ff3cc8", lw=2.8, zorder=7))
        L_ = hud_lines(spec, st, f); cols = ["white", "#aaa", "white", "#ff3cc8"]; ys = [.93, .70, .47, .24]
        for t, colr, y in zip(L_, cols, ys):
            if t: self._add(self.at.text(0, y, t, color=colr, fontsize=10.5 if y == .93 else 9, weight="bold" if y == .93 else "normal", va="top"))
        self._add(self.at.text(0, .04, "orange = focus team  blue = opponent  |  red/amber/green = lane blocked/contested/open  |  cyan = defensive line", color="#777", fontsize=6.5, va="top"))
        self.fig.canvas.draw()
        return np.asarray(self.fig.canvas.buffer_rgba())[..., :3][..., ::-1].copy()

def _fit(img, W_, H_):
    h, w = img.shape[:2]; s = min(W_ / w, H_ / h); nw, nh = int(w * s), int(h * s)
    out = np.zeros((H_, W_, 3), np.uint8); y0, x0 = (H_ - nh) // 2, (W_ - nw) // 2
    out[y0:y0 + nh, x0:x0 + nw] = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA); return out

def _to_h264(path):
    if shutil.which("ffmpeg") is None: return
    tmp = path.with_name(path.stem + "_h264.mp4")
    if subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(path), "-vcodec", "libx264", "-pix_fmt", "yuv420p", "-crf", "22", str(tmp)]).returncode == 0:
        path.unlink(); tmp.rename(path)

def render_clip(spec, cap):
    out = CLIPS_DIR / f"{spec['clip_id']}.mp4"
    if out.exists() and not OVERWRITE:
        return out, None, "skipped (exists)"
    states = make_states(spec)
    if not states: return None, None, "no player data in window"
    panel = Panel(states, spec) if RENDER_MODE in ("both", "tactical") else None
    ow = {"both": VIDEO_W + PANEL_W, "overlay": VIDEO_W, "tactical": PANEL_W}[RENDER_MODE]
    wr = cv2.VideoWriter(str(out), cv2.VideoWriter_fourcc(*"mp4v"), FPS / OUT_STRIDE, (ow, OUT_H)); key, best = None, 10 ** 9
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(spec["fs"]) + VIDEO_FRAME_OFFSET)
    for f in range(int(spec["fs"]), int(spec["fe"]) + 1):
        ok, img = cap.read()
        if not ok: break
        if (f - spec["fs"]) % OUT_STRIDE: continue
        st = states.get(f); parts = []
        if RENDER_MODE in ("both", "overlay"): parts.append(_fit(draw_overlay(img, st, f, spec), VIDEO_W, OUT_H))
        if panel is not None: parts.append(panel.render(f))
        fr_ = np.hstack(parts); wr.write(fr_)
        if abs(f - spec["anchor"]) < best: best, key = abs(f - spec["anchor"]), fr_.copy()
    wr.release(); _to_h264(out)
    if key is not None: cv2.imwrite(str(CLIPS_DIR / f"{spec['clip_id']}.jpg"), key)
    return out, key, "ok"

# ---------------- run ----------------
assert len(CLIP_PLAN), "empty CLIP_PLAN - run 19c first"
cap = cv2.VideoCapture(str(VIDEO_PATH)); assert cap.isOpened(), f"cannot open {VIDEO_PATH}"
print(f"video {int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))}x{int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))} @ {cap.get(cv2.CAP_PROP_FPS):.2f} fps, "
      f"{int(cap.get(cv2.CAP_PROP_FRAME_COUNT))} frames (table: {int(player_frame_table.frame_idx.max()) + 1})")
_idx, _prev = [], None
for i, spec in enumerate(CLIP_PLAN.to_dict("records"), 1):
    t0 = time.time(); path, key, status = render_clip(spec, cap)
    print(f"[{i}/{len(CLIP_PLAN)}] {spec['clip_id']}  {status}  ({time.time() - t0:.0f}s)")
    if path is not None:
        _idx.append(dict(clip_id=spec["clip_id"], file=path.name, family=spec["family"], subtype=spec["subtype"], team=spec["team"], team_name=TEAM_NAMES[spec["team"]],
                         group=spec["group"], zone=spec["zone"], channel=spec["channel"], target=spec["target"], anchor_frame=spec["anchor"],
                         start_sec=round(spec["fs"] / FPS, 1), end_sec=round(spec["fe"] / FPS, 1), clock=_clock_f(spec["anchor"]), score=round(float(spec["score"]), 2)))
cap.release()
CLIP_INDEX = pd.DataFrame(_idx)
CLIP_INDEX.to_csv(EXPORT_DIR / "clip_index.csv", index=False)
print(f"\nclip index ({len(CLIP_INDEX)} clips) -> {EXPORT_DIR / 'clip_index.csv'}\nclips in: {CLIPS_DIR}")

# contact sheet: the anchor frame of every clip
_keys = [(r.clip_id, cv2.imread(str(CLIPS_DIR / f"{r.clip_id}.jpg"))) for r in CLIP_INDEX.itertuples()]
_keys = [(n, k) for n, k in _keys if k is not None]
if _keys:
    cols = 2; rows_ = int(np.ceil(len(_keys) / cols)); fig, axs = plt.subplots(rows_, cols, figsize=(18, 4.9 * rows_)); axs = np.atleast_1d(axs).ravel()
    for a in axs: a.axis("off")
    for a, (n, k) in zip(axs, _keys): a.imshow(k[..., ::-1]); a.set_title(n, fontsize=8)
    plt.tight_layout(); plt.show()


## 19e. Stoppage verification sheets + clip index
Stoppages (section 0b) are detected from carrier gaps, not seen - so each one gets a contact sheet (1 s before the start, the start, the end, 1 s after) to check by eye that the detector flagged dead time and nothing live. Saved to `pressure_analysis/stoppage_check/`. The clip index from 19d is what the exports (sections 20 / 21) pick up.


In [ ]:
# ===== 19e. Stoppage verification sheets =====
STOP_DIR = EXPORT_DIR / "stoppage_check"; STOP_DIR.mkdir(exist_ok=True)
cap = cv2.VideoCapture(str(VIDEO_PATH)); _n = 0
for k, s in stoppages.reset_index(drop=True).iterrows():
    sheet = []
    for lab, f in [("1s before", int(s.start_frame) - FPS), ("start", int(s.start_frame)), ("end", int(s.end_frame)), ("1s after", int(s.end_frame) + FPS)]:
        cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, f) + VIDEO_FRAME_OFFSET); ok, img = cap.read()
        if not ok: img = np.zeros((360, 640, 3), np.uint8)
        img = cv2.resize(img, (640, 360)); cv2.putText(img, f"{lab}  f{f}  {_clock_f(f)}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2, cv2.LINE_AA); sheet.append(img)
    cv2.imwrite(str(STOP_DIR / f"stoppage_{k:02d}_{_clock_f(int(s.start_frame)).replace(':', 'm')}s_{s.duration_sec:.0f}s.jpg"), np.vstack([np.hstack(sheet[:2]), np.hstack(sheet[2:])])); _n += 1
cap.release()
print(f"{_n} stoppage sheets written to {STOP_DIR}")


## Hand-off
Saves what the later notebooks need.

In [ ]:
save_state(globals(), [
    'CLIP_INDEX',
], notebook='06_clips.ipynb')